In [1]:
!pip install gradio -q

Import libraries

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

import gradio as gr

Load dataset

In [3]:
df = pd.read_csv("/content/fraudTest.csv")
df.head()

,Unnamed: 0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,...,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
0,0,2020-06-21 12:14:25,2291163933867244,fraud_Kirlin and Sons,personal_care,2.86,Jeff,Elliott,M,351 Darlene Green,...,33.9659,-80.9355,333497,Mechanical engineer,1968-03-19,2da90c7d74bd46a0caf3777415b3ebd3,1371816865,33.986391,-81.200714,0
1,1,2020-06-21 12:14:33,3573030041201292,fraud_Sporer-Keebler,personal_care,29.84,Joanne,Williams,F,3638 Marsh Union,...,40.3207,-110.4360,302,"Sales professional, IT",1990-01-17,324cc204407e99f51b0d6ca0055005e7,1371816873,39.450498,-109.960431,0
2,2,2020-06-21 12:14:53,3598215285024754,"fraud_Swaniawski, Nitzsche and Welch",health_fitness,41.28,Ashley,Lopez,F,9333 Valentine Point,...,40.6729,-73.5365,34496,"Librarian, public",1970-10-21,c81755dbbbea9d5c77f094348a7579be,1371816893,40.495810,-74.196111,0
3,3,2020-06-21 12:15:15,3591919803438423,fraud_Haley Group,misc_pos,60.05,Brian,Williams,M,32941 Krystal Mill Apt. 552,...,28.5697,-80.8191,54767,Set designer,1987-07-25,2159175b9efe66dc301f149d3d5abf8c,1371816915,28.812398,-80.883061,0
4,4,2020-06-21 12:15:17,3526826139003047,fraud_Johnston-Casper,travel,3.19,Nathan,Massey,M,5783 Evan Roads Apt. 465,...,44.2529,-85.0170,1126,Furniture designer,1955-07-06,57ff021bd3f328f8738bb535c302a31b,1371816917,44.959148,-85.884734,0


Check dataset

In [5]:
print(df.shape)
print(df.isnull().sum())
print(df["is_fraud"].value_counts())

(555719, 23)
Unnamed: 0               0
trans_date_trans_time    0
cc_num                   0
merchant                 0
category                 0
amt                      0
first                    0
last                     0
gender                   0
street                   0
city                     0
state                    0
zip                      0
lat                      0
long                     0
city_pop                 0
job                      0
dob                      0
trans_num                0
unix_time                0
merch_lat                0
merch_long               0
is_fraud                 0
dtype: int64
is_fraud
0    553574
1      2145
Name: count, dtype: int64


Split data

In [7]:
X = df.drop("is_fraud", axis=1)
y = df["is_fraud"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

Scale data

In [9]:
non_numeric_cols = ['Unnamed: 0', 'trans_date_trans_time', 'cc_num', 'merchant', 'category', 'first', 'last', 'gender', 'street', 'city', 'state', 'zip', 'job', 'dob', 'trans_num']

X_train_numeric = X_train.drop(columns=non_numeric_cols, errors='ignore')
X_test_numeric = X_test.drop(columns=non_numeric_cols, errors='ignore')

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_numeric)
X_test_scaled = scaler.transform(X_test_numeric)
print("Scaling completed")

Scaling completed


Train Logistic Regression

In [10]:
lr_model = LogisticRegression(max_iter=1000, class_weight="balanced")
lr_model.fit(X_train_scaled, y_train)
lr_pred = lr_model.predict(X_test_scaled)
print("Logistic Regression Accuracy:", accuracy_score(y_test, lr_pred))
print(classification_report(y_test, lr_pred))

Logistic Regression Accuracy: 0.950154754192759
              precision    recall  f1-score   support

           0       1.00      0.95      0.97    110715
           1       0.06      0.74      0.10       429

    accuracy                           0.95    111144
   macro avg       0.53      0.84      0.54    111144
weighted avg       1.00      0.95      0.97    111144



Train Decision Tree

In [14]:
dt_model = DecisionTreeClassifier(random_state=42, class_weight="balanced")
dt_model.fit(X_train_scaled, y_train)
dt_pred = dt_model.predict(X_test_scaled)

print("Decision Tree Accuracy:", accuracy_score(y_test, dt_pred))
print(classification_report(y_test, dt_pred))

Decision Tree Accuracy: 0.9957802490462823
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    110715
           1       0.45      0.41      0.43       429

    accuracy                           1.00    111144
   macro avg       0.72      0.70      0.71    111144
weighted avg       1.00      1.00      1.00    111144



Train Random Forest

In [12]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced"
)

rf_model.fit(X_train_scaled, y_train)

rf_pred = rf_model.predict(X_test_scaled)

print("Random Forest Accuracy:", accuracy_score(y_test, rf_pred))
print(classification_report(y_test, rf_pred))

Random Forest Accuracy: 0.9976606924350392
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    110715
           1       0.90      0.44      0.59       429

    accuracy                           1.00    111144
   macro avg       0.95      0.72      0.80    111144
weighted avg       1.00      1.00      1.00    111144



Compare all accuracy

In [13]:
print("===== MODEL ACCURACY COMPARISON =====")
print("Logistic Regression:", accuracy_score(y_test, lr_pred))
print("Decision Tree:", accuracy_score(y_test, dt_pred))
print("Random Forest:", accuracy_score(y_test, rf_pred))

===== MODEL ACCURACY COMPARISON =====
Logistic Regression: 0.950154754192759
Decision Tree: 0.9957802490462823
Random Forest: 0.9976606924350392


In [15]:
cm = confusion_matrix(y_test, rf_pred)

print(cm)

[[110694     21]
 [   239    190]]


UI

In [23]:
!pip install gradio -q

import pandas as pd
import numpy as np
import gradio as gr

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Load dataset
df = pd.read_csv("/content/fraudTest.csv") # Corrected dataset path

# Remove useless columns
df = df.drop(columns=["Unnamed: 0", "trans_date_trans_time", "first", "last", "street", "dob", "trans_num"], errors="ignore")

# Convert text columns into numbers
le = LabelEncoder()
object_cols_for_le = df.select_dtypes(include="object").columns.tolist()
for col in object_cols_for_le:
    df[col] = le.fit_transform(df[col].astype(str))

# Features and target
X = df.drop("is_fraud", axis=1)
y = df["is_fraud"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y # Added stratify=y
)

# Train model
model = RandomForestClassifier(n_estimators=30, random_state=42)
model.fit(X_train, y_train)

# Accuracy
pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, pred))

# UI prediction
feature_columns = X.columns.tolist() # Get all feature columns the model was trained with

def predict_fraud(*input_values):
    # Create a DataFrame from the input values, ensuring correct column order
    input_data_dict = dict(zip(feature_columns, input_values))
    data = pd.DataFrame([input_data_dict])

    # Predict
    prediction = model.predict(data)[0]

    if prediction == 1:
        return "🚨 Fraud Transaction Detected"
    else:
        return "✅ Legitimate Transaction"

# Gradio UI
gr_inputs = []
# Provide reasonable default values for input fields
default_values = {}
for col in feature_columns:
    if col == 'amt':
        default_values[col] = 50.0
    elif col in ['lat', 'long', 'merch_lat', 'merch_long']:
        default_values[col] = 0.0 # Example geographic center
    elif col == 'city_pop':
        default_values[col] = 10000
    elif col == 'unix_time':
        default_values[col] = 1600000000 # Example recent timestamp
    elif col in object_cols_for_le: # For label-encoded columns
        default_values[col] = 0 # Default to the first encoded category
    else: # For other numericals like cc_num, zip (assuming they are now numeric)
        default_values[col] = 0

for feature in feature_columns:
    gr_inputs.append(gr.Number(label=feature.replace('_', ' ').title(), value=default_values.get(feature, 0)))

app = gr.Interface(
    fn=predict_fraud,
    inputs=gr_inputs, # Use dynamically generated inputs
    outputs=gr.Textbox(label="Prediction Result"),
    title="💳 Credit Card Fraud Detection (FraudTest.csv - Label Encoded)",
    description="ML UI with RandomForestClassifier using LabelEncoder for categorical features."
)

app.launch(share=True)

Accuracy: 0.998263513999856
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c963f03dda0ade4059.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
